# Data Preprocessing 

This notebook completes roadmap task 5. It prepares the three selected datasets as separate, typed Parquet tables for database storage and later integration. The original [data exploration notebook](data_exploration.ipynb) is unchanged.

Roadmap task 4 is documented in [five analytical questions](../docs/analytical_questions.md). The [preprocessing guide](../docs/preprocessing.md) explains the output contract for teammates.

## 1. Setup and source selection

The exploration selected **cost-of-living_v2.csv** because it has broader geographic coverage. We do not concatenate both versions, which would double count locations and mix source snapshots. The OECD file has two metadata rows before its CSV header. Property data is the supplied 2026 country-level file.

In [1]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
from IPython.display import display

# Works when Jupyter starts in the repository root or notebooks directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'roadmap.txt').is_file() and (p / 'data/raw').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from inside the project directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    SOURCE_FILES, clean_cost_of_living, clean_income,
    clean_property_prices, save_cleaned,
)
from src.cost_columns import COST_COLUMNS

RAW_DIR = PROJECT_ROOT / 'data/raw'
OUTPUT_DIR = PROJECT_ROOT / 'data/cleaned'
raw_hashes_before = {
    p.relative_to(RAW_DIR).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(RAW_DIR.rglob('*.csv'))
}
display(pd.DataFrame([
    {'dataset': name, 'source': relative, 'file_exists': (RAW_DIR / relative).is_file()}
    for name, relative in SOURCE_FILES.items()
]))

,dataset,source,file_exists
0,cost_of_living,cost-of-living/cost-of-living_v2.csv,True
1,income,income/export-2026-09-21T08_03_15.042Z.csv,True
2,property_prices,real-estate/untera-global-price-per-sqm-2026.csv,True


## 2. Cleaning decisions

| Issue | Decision |
| --- | --- |
| Missing values | Keep nulls. Do not fill with zero or a global median. Preserve rows with no measures and mark `has_usable_measures = False`. Later analyses must require their own inputs. |
| Duplicates | Remove exact source records. Stop on conflicting records with the same normalized key. |
| Invalid numbers | Set nonnumeric, nonfinite, or negative values to null and record each change. Country wages, property prices, source ranks, and listing counts must be positive; counts/ranks must be integers. |
| Zeros | Preserve zero cost values and flag them for review. Free services and missing prices are different. |
| Outliers | Flag values outside global 1.5-IQR bounds per measure, with at least four observations and a nonzero IQR. Keep the values; statistical flags do not prove error. |
| Country identity | Use explicit aliases and ISO alpha-2 codes. Resolve ambiguous Congo rows by city. Mark Kosovo's XK as user-assigned. |
| Units and years | Preserve nominal USD vs constant PPP USD. Keep the cost observation year unknown. Mortgage rates remain percent, not USD. |
| Source quality | Retain both `data_quality` values. Primary analyses can use 1 and show a sensitivity view including 0. |

Functions live in `src/preprocessing.py` so notebook and command-line runs apply the same rules. Country matching is in `src/countries.py`. No city averaging, cross-dataset joins, affordability ratios, or imputation happen at this stage.

## 3. Clean city cost-of-living data

Rename `x1`–`x55` using the variable dictionary from exploration, preserve the original country label and source row, and assign numeric types. The cost snapshot year is not supplied, so `reference_year` is null. `mortgage_interest_rate_percent` is the only nonmonetary measure in this dictionary.

In [2]:
cost = clean_cost_of_living(RAW_DIR / SOURCE_FILES['cost_of_living'])
display(pd.DataFrame({
    'source_column': list(COST_COLUMNS),
    'cleaned_column': list(COST_COLUMNS.values()),
}).tail(10))
display(cost.frame[[
    'city', 'country', 'country_code', 'rent_1bed_outside_centre_monthly_usd',
    'average_monthly_net_salary_usd', 'data_quality', 'missing_measure_count',
]].head(10))

,source_column,cleaned_column
45,x46,nike_running_shoes_usd
46,x47,mens_business_shoes_usd
47,x48,rent_1bed_city_centre_monthly_usd
48,x49,rent_1bed_outside_centre_monthly_usd
49,x50,rent_3bed_city_centre_monthly_usd
50,x51,rent_3bed_outside_centre_monthly_usd
51,x52,buy_price_sqm_city_centre_usd
52,x53,buy_price_sqm_outside_centre_usd
53,x54,average_monthly_net_salary_usd
54,x55,mortgage_interest_rate_percent


,city,country,country_code,rent_1bed_outside_centre_monthly_usd,average_monthly_net_salary_usd,data_quality,missing_measure_count
0,Andorra la Vella,Andorra,AD,645.42,2509.67,0,1
1,Escaldes-Engordany,Andorra,AD,790.31,2574.65,0,9
2,Sant Julia de Loria,Andorra,AD,558.48,3442.23,0,9
3,Abu Dhabi,United Arab Emirates,AE,1058.28,3541.4,1,0
4,Al Fujayrah,United Arab Emirates,AE,418.66,3378.16,1,0
5,Dubai,United Arab Emirates,AE,1015.88,4322.14,1,0
6,Sharjah,United Arab Emirates,AE,4996.72,1681.97,1,0
7,`Ajman,United Arab Emirates,AE,314.8,3630.09,1,0
8,Gardez,Afghanistan,AF,<NA>,<NA>,0,43
9,Herat,Afghanistan,AF,39.55,152.54,0,7


In [3]:
missing = pd.DataFrame({
    'missing_before': cost.report['missing_before'],
    'missing_after': cost.report['missing_after'],
})
missing['missing_percent'] = (100 * missing['missing_after'] / len(cost.frame)).round(2)
display(missing.sort_values('missing_after', ascending=False).head(15))
display(pd.DataFrame([{
    'input_rows': cost.report['input_rows'],
    'output_rows': cost.report['output_rows'],
    'exact_duplicates_removed': cost.report['exact_duplicates_removed'],
    'missing_measure_values': int(missing['missing_after'].sum()),
    'rows_without_measures': cost.report['rows_without_measures'],
}]))
display(cost.frame['data_quality'].value_counts(dropna=False).rename('rows').to_frame())

,missing_before,missing_after,missing_percent
tennis_court_1hour_usd,2400,2400,48.43
buy_price_sqm_outside_centre_usd,2303,2303,46.47
buy_price_sqm_city_centre_usd,2227,2227,44.94
monthly_transport_pass_usd,2166,2166,43.70
international_primary_school_yearly_usd,1681,1681,33.92
rent_3bed_outside_centre_monthly_usd,1512,1512,30.51
one_way_transport_ticket_usd,1506,1506,30.39
rent_3bed_city_centre_monthly_usd,1480,1480,29.86
average_monthly_net_salary_usd,1432,1432,28.89
rent_1bed_outside_centre_monthly_usd,1431,1431,28.87


,input_rows,output_rows,exact_duplicates_removed,missing_measure_values,rows_without_measures
0,4956,4956,0,45858,50


,rows
data_quality,
0,4033
1,923


### Country names and ambiguous labels

The same country can be named differently across sources: Korea/South Korea, Czech Republic/Czechia, Slovak Rep./Slovakia, and UAE/United Arab Emirates. Country codes provide the join key. The source label **Congo** contains cities in two countries, so a single name replacement would produce wrong joins. `source_country` records the original label. Country coverage includes territories and should not be described as sovereign-country coverage.

In [4]:
display(cost.frame.loc[
    cost.frame['source_country'].isin(['Congo', 'Kosovo (Disputed Territory)', 'Czech Republic']),
    ['city', 'source_country', 'country', 'country_code', 'country_code_status'],
].head(12))
print('Distinct source country/territory labels:', cost.frame['source_country'].nunique())
print('Standardized country/territory codes:', cost.frame['country_code'].nunique())

,city,source_country,country,country_code,country_code_status
579,Goma,Congo,Democratic Republic of the Congo,CD,ISO 3166-1 alpha-2
580,Kinshasa,Congo,Democratic Republic of the Congo,CD,ISO 3166-1 alpha-2
581,Lubumbashi,Congo,Democratic Republic of the Congo,CD,ISO 3166-1 alpha-2
582,Brazzaville,Congo,Republic of the Congo,CG,ISO 3166-1 alpha-2
583,Pointe-Noire,Congo,Republic of the Congo,CG,ISO 3166-1 alpha-2
815,Bilovec,Czech Republic,Czechia,CZ,ISO 3166-1 alpha-2
816,Brno,Czech Republic,Czechia,CZ,ISO 3166-1 alpha-2
817,Ceske Budejovice,Czech Republic,Czechia,CZ,ISO 3166-1 alpha-2
818,Cesky Krumlov,Czech Republic,Czechia,CZ,ISO 3166-1 alpha-2
819,Chomutov,Czech Republic,Czechia,CZ,ISO 3166-1 alpha-2


Distinct source country/territory labels: 215
Standardized country/territory codes: 216


### Outlier and zero-value review

Global IQR flags are intentionally only a review aid: real price differences between places can be large. Extreme source values, such as Honiara's reported internet price, remain visible with flags. No winsorization or automatic outlier deletion is applied. Analysts should inspect relevant flags before publishing a ranking.

In [5]:
outliers = pd.Series(cost.report['outliers_by_column'], name='flagged_values')
display(outliers.sort_values(ascending=False).head(10).to_frame())
display(cost.flags.loc[
    cost.flags['city'].isin(['Honiara', 'Niamey', 'Kermanshah']),
    ['city', 'country_code', 'column', 'issue', 'source_value'],
].head(15))
display(pd.Series(cost.report['zero_values_by_column'], name='zero_values').loc[lambda s: s > 0].to_frame())

,flagged_values
rice_1kg_usd,263
banana_1kg_usd,225
volkswagen_golf_usd,223
mortgage_interest_rate_percent,215
milk_1l_usd,214
private_preschool_monthly_usd,211
oranges_1kg_usd,186
beef_1kg_usd,185
toyota_corolla_usd,183
potato_1kg_usd,178


,city,country_code,column,issue,source_value
94,Honiara,SB,mcmeal_usd,global_iqr_outlier_review,15.19
425,Niamey,NE,milk_1l_usd,global_iqr_outlier_review,2.45
439,Honiara,SB,milk_1l_usd,global_iqr_outlier_review,2.67
1855,Honiara,SB,oranges_1kg_usd,global_iqr_outlier_review,6.07
2118,Honiara,SB,potato_1kg_usd,global_iqr_outlier_review,4.5
3523,Niamey,NE,taxi_1hour_waiting_usd,global_iqr_outlier_review,160.64
3624,Niamey,NE,volkswagen_golf_usd,zero_value_review,0.0
3657,Niamey,NE,volkswagen_golf_usd,global_iqr_outlier_review,0.0
3668,Kermanshah,IR,volkswagen_golf_usd,global_iqr_outlier_review,66211.5
3882,Kermanshah,IR,toyota_corolla_usd,global_iqr_outlier_review,48771.0


,zero_values
one_way_transport_ticket_usd,46
monthly_transport_pass_usd,67
volkswagen_golf_usd,1
mobile_tariff_1min_usd,55
fitness_monthly_usd,1
tennis_court_1hour_usd,97
international_primary_school_yearly_usd,1
summer_dress_usd,1


## 4. Clean OECD country wages

Read and validate the two metadata lines. Remove the **OECD aggregate**, which is not a country. Keep `annual_wage_usd_ppp` and derive `monthly_wage_usd_ppp = annual_wage_usd_ppp / 12`. Both remain **constant 2025 USD converted using private-consumption PPPs**, and wages are gross full-time-equivalent averages. Monthly conversion does not turn PPP wages into nominal or net salary. Never divide nominal USD rent by these wages.

In [6]:
income = clean_income(RAW_DIR / SOURCE_FILES['income'])
display(pd.DataFrame([{
    'input_rows': income.report['input_rows'],
    'excluded_OECD_aggregate_rows': income.report['excluded_aggregate_rows'],
    'output_country_rows': income.report['output_rows'],
    'exact_duplicates_removed': income.report['exact_duplicates_removed'],
}]))
display(income.frame[[
    'source_country', 'country', 'country_code', 'annual_wage_usd_ppp',
    'monthly_wage_usd_ppp', 'reference_year', 'monetary_basis',
]].head(10))

,input_rows,excluded_OECD_aggregate_rows,output_country_rows,exact_duplicates_removed
0,35,1,34,0


,source_country,country,country_code,annual_wage_usd_ppp,monthly_wage_usd_ppp,reference_year,monetary_basis
0,Austria,Austria,AT,78301.337,6525.111417,2025,constant_2025_usd_ppp_private_consumption
1,Australia,Australia,AU,72018.439,6001.536583,2025,constant_2025_usd_ppp_private_consumption
2,Belgium,Belgium,BE,80008.652,6667.387667,2025,constant_2025_usd_ppp_private_consumption
3,Canada,Canada,CA,67900.601,5658.383417,2025,constant_2025_usd_ppp_private_consumption
4,Switzerland,Switzerland,CH,92285.059,7690.421583,2025,constant_2025_usd_ppp_private_consumption
5,Czechia,Czechia,CZ,43607.144,3633.928667,2025,constant_2025_usd_ppp_private_consumption
6,Germany,Germany,DE,76284.517,6357.043083,2025,constant_2025_usd_ppp_private_consumption
7,Denmark,Denmark,DK,78090.445,6507.537083,2025,constant_2025_usd_ppp_private_consumption
8,Estonia,Estonia,EE,41019.039,3418.25325,2025,constant_2025_usd_ppp_private_consumption
9,Spain,Spain,ES,57778.662,4814.8885,2025,constant_2025_usd_ppp_private_consumption


## 5. Clean country property prices

Validate the supplied country codes against standardized names. Preserve positive `usd_per_sqm`, integer `rank`, and integer `listings_sampled`. These are **country median residential asking prices**, with a June 2026 reference snapshot according to the publisher. They do not represent Vienna-specific prices or completed transactions. [Untera source documentation](https://github.com/PataMills/global-property-price-data).

In [7]:
property_prices = clean_property_prices(RAW_DIR / SOURCE_FILES['property_prices'])
display(property_prices.frame[[
    'country', 'country_code', 'usd_per_sqm', 'listings_sampled',
    'rank', 'reference_year', 'price_basis',
]].head(10))
display(pd.DataFrame([{
    'input_rows': property_prices.report['input_rows'],
    'output_rows': property_prices.report['output_rows'],
    'exact_duplicates_removed': property_prices.report['exact_duplicates_removed'],
    'missing_property_prices': property_prices.report['missing_after']['usd_per_sqm'],
}]))

,country,country_code,usd_per_sqm,listings_sampled,rank,reference_year,price_basis
0,United Arab Emirates,AE,4533.0,112493,58,2026,country_median_asking_price_per_sqm
1,Argentina,AR,1850.0,9893,29,2026,country_median_asking_price_per_sqm
2,Austria,AT,5133.0,15706,60,2026,country_median_asking_price_per_sqm
3,Bangladesh,BD,830.0,1036,7,2026,country_median_asking_price_per_sqm
4,Belgium,BE,2731.0,27302,41,2026,country_median_asking_price_per_sqm
5,Bahrain,BH,2647.0,12232,38,2026,country_median_asking_price_per_sqm
6,Bolivia,BO,881.0,9000,10,2026,country_median_asking_price_per_sqm
7,Brazil,BR,1342.0,225205,19,2026,country_median_asking_price_per_sqm
8,Canada,CA,3489.0,31745,49,2026,country_median_asking_price_per_sqm
9,Chile,CL,1412.0,2307,23,2026,country_median_asking_price_per_sqm


,input_rows,output_rows,exact_duplicates_removed,missing_property_prices
0,62,62,0,0


## 6. Validate keys, units, and coverage

Cost data has one row per city and country code. Wages and property prices each have one row per country code. Check country overlap for the later integration task, without dropping countries that are useful for other analyses. Keep the national and city geographic levels explicit.

In [8]:
datasets = {'cost_of_living': cost, 'income': income, 'property_prices': property_prices}
assert not cost.frame.duplicated(['country_code', 'city']).any()
assert not income.frame['country_code'].duplicated().any()
assert not property_prices.frame['country_code'].duplicated().any()
assert cost.frame['reference_year'].isna().all()
assert income.frame['reference_year'].eq(2025).all()
assert property_prices.frame['reference_year'].eq(2026).all()
assert income.frame['monetary_basis'].ne('nominal_usd').all()
pd.testing.assert_series_equal(
    income.frame['monthly_wage_usd_ppp'], income.frame['annual_wage_usd_ppp'] / 12,
    check_names=False,
)
common_codes = set(cost.frame['country_code']) & set(income.frame['country_code']) & set(property_prices.frame['country_code'])
print('Country codes available in all three datasets:', len(common_codes))
display(income.frame.loc[income.frame['country_code'].isin(common_codes), ['country', 'country_code']].reset_index(drop=True))
display(pd.DataFrame([
    {'dataset': name, 'rows': len(item.frame), 'country_codes': item.frame['country_code'].nunique(),
     'invalid_values_set_null': sum(item.report['invalid_values_set_null'].values()),
     'rows_with_outliers': item.report['rows_with_outliers']}
    for name, item in datasets.items()
]))

Country codes available in all three datasets: 23


,country,country_code
0,Austria,AT
1,Belgium,BE
2,Canada,CA
3,Czechia,CZ
4,Germany,DE
5,Denmark,DK
6,Spain,ES
7,Finland,FI
8,France,FR
9,United Kingdom,GB


,dataset,rows,country_codes,invalid_values_set_null,rows_with_outliers
0,cost_of_living,4956,216,0,2360
1,income,34,34,0,0
2,property_prices,62,62,0,2


## 7. Export Parquet and the quality report

Write three separate cleaned data tables, `quality_flags.parquet` (one row per flagged cell/issue), and `quality_report.json` (counts, nulls, types, IQR bounds, source hashes, aliases' resulting codes, and the variable dictionary). Flags use the original CSV row number so each issue can be traced back. The JSON has no execution timestamp, making repeated runs directly comparable.

In [9]:
report = save_cleaned(datasets, OUTPUT_DIR)
display(pd.DataFrame([
    {'file': p.name, 'size_bytes': p.stat().st_size}
    for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()
]))
print('Review flags:', report['quality_flag_count'])
print('Countries in all three:', report['countries_in_all_three_count'])

,file,size_bytes
0,cost_of_living.parquet,742255
1,income.parquet,12222
2,property_prices.parquet,13804
3,quality_flags.parquet,99324
4,quality_report.json,32478


Review flags: 6664
Countries in all three: 23


## 8. Verify the saved files and original sources

Read each Parquet table back and compare values and dtypes to the prepared DataFrame. Check that all raw CSV bytes are unchanged. This verifies the actual exported files, not just the in-memory results.

In [10]:
for name, item in datasets.items():
    restored = pd.read_parquet(OUTPUT_DIR / f'{name}.parquet')
    pd.testing.assert_frame_equal(restored, item.frame)
    print(f'{name}: Parquet values and dtypes verified ({len(restored):,} rows)')
saved_flags = pd.read_parquet(OUTPUT_DIR / 'quality_flags.parquet')
assert len(saved_flags) == report['quality_flag_count']
raw_hashes_after = {
    p.relative_to(RAW_DIR).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(RAW_DIR.rglob('*.csv'))
}
assert raw_hashes_before == raw_hashes_after
print('All four raw CSV files are unchanged.')

cost_of_living: Parquet values and dtypes verified (4,956 rows)
income: Parquet values and dtypes verified (34 rows)
property_prices: Parquet values and dtypes verified (62 rows)
All four raw CSV files are unchanged.


## Handoff

The cleaned tables are ready for the database and data-lake tasks. Join country context with a validated many-to-one relationship on `country_code`; do not treat country figures as city measurements. Use only the nonmissing fields required by each analytical question, expose source quality and relevant outlier flags, and preserve reference dates and monetary bases.

The analytical questions document also records the unresolved second-data-source requirement for the travel story. The supplied datasets cannot establish a complete trip or home-ownership budget. Integration, DuckDB analysis, and the dashboard remain later roadmap tasks.